# Paso 1: Identificar el Universo de Modelado

In [0]:
df = spark.table("data.client_transaction_orders")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, col, lit, datediff, max

# 1. Obtenemos la última transacción de cada cliente usando una función de ventana
windowSpec = Window.partitionBy("cliente_id").orderBy(desc("fecha_pedido_dt"))
df_ultima_transaccion = df.withColumn("rank", row_number().over(windowSpec)).filter("rank = 1")

# 2. Filtramos para quedarnos con aquellos cuya última transacción NO fue digital
ultimas_no_digitales = df_ultima_transaccion.filter(col("canal_pedido_cd") != "DIGITAL")

# 3. Calculamos la recencia de estos clientes
# Primero, obtenemos la fecha más reciente del dataset como referencia
fecha_referencia = df.agg(max("fecha_pedido_dt")).collect()[0][0]

# Ahora, calculamos los días desde su última compra
clientes_recientes_no_digitales = ultimas_no_digitales.withColumn(
    "recencia",
    datediff(lit(fecha_referencia), col("fecha_pedido_dt"))
)

# 4. Filtramos finalmente por recencia para obtener el grupo objetivo (ej. compraron en los últimos 60 días)
clientes_para_convertir = clientes_recientes_no_digitales.filter(col("recencia") <= 60)


# Mostramos el resultado y el total de clientes en este segmento
print(f"Total de clientes recientes para convertir a digital: {clientes_para_convertir.count()}")
display(clientes_para_convertir)

In [0]:
from pyspark.sql.functions import collect_set, size, col, min, lit
# Identificamos los clientes multicanal (han usado >1 canal)
canales_por_cliente = df.groupBy("cliente_id").agg(collect_set("canal_pedido_cd").alias("canales_usados"))
clientes_multicanal_ids = canales_por_cliente.filter(size(col("canales_usados")) > 1).select("cliente_id")

# Identificamos los clientes recientes para convertir (activos, última compra no digital)
# (Este es el grupo al que querremos aplicarle el modelo)
clientes_para_convertir_ids = clientes_para_convertir.select("cliente_id") # Usamos el DataFrame del análisis RFM

# Filtramos las transacciones para quedarnos solo con el universo de clientes multicanal
df_modelo = df.join(clientes_multicanal_ids, on="cliente_id", how="inner")

# Paso 2: Ingeniería de Features

In [0]:
from pyspark.sql.functions import sum, count, max, avg, datediff, first, when

# Usamos la última fecha del dataset como referencia
fecha_referencia = lit('2024-08-23')

# Calculamos features basadas en el historial completo
features_cliente = df_modelo.groupBy("cliente_id").agg(
    # Métricas RFM
    datediff(fecha_referencia, max("fecha_pedido_dt")).alias("recencia"),
    count("*").alias("frecuencia_total"),
    sum("facturacion_usd_val").alias("monto_total"),
    
    # Métricas de comportamiento de canal
    sum(when(col("canal_pedido_cd") == "DIGITAL", 1).otherwise(0)).alias("frecuencia_digital"),
    sum(when(col("canal_pedido_cd") != "DIGITAL", 1).otherwise(0)).alias("frecuencia_no_digital"),
    
    # Métricas de perfil de compra
    avg("materiales_distintos_val").alias("promedio_materiales_distintos"),
    
    # Atributos del cliente (tomamos el último valor conocido)
    first("tipo_cliente_cd", ignorenulls=True).alias("tipo_cliente_cd"),
    first("madurez_digital_cd", ignorenulls=True).alias("madurez_digital_cd"),
    first("estrellas_txt", ignorenulls=True).alias("estrellas_txt"),
    first("pais_cd", ignorenulls=True).alias("pais_cd")
)

# Calculamos el porcentaje de uso digital
features_cliente = features_cliente.withColumn(
    "porcentaje_digital",
    (col("frecuencia_digital") / col("frecuencia_total")) * 100
)

# Paso 3: Definir la Variable Objetivo 

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# Obtenemos la última transacción de cada cliente del universo de modelado
windowSpec = Window.partitionBy("cliente_id").orderBy(desc("fecha_pedido_dt"))
df_ultima_transaccion = df_modelo.withColumn("rank", row_number().over(windowSpec)).filter("rank = 1")

# Creamos la variable objetivo (target)
target = df_ultima_transaccion.select(
    "cliente_id",
    when(col("canal_pedido_cd") == "DIGITAL", 1).otherwise(0).alias("target_prox_trx_digital")
)

In [0]:
# Unimos las features con el target
dataset_final_para_modelar = features_cliente.join(target, on="cliente_id", how="inner")


In [0]:


# Guardamos el DataFrame como una tabla en el catálogo especificado
training_set, validation_set = dataset_final_para_modelar.randomSplit([0.8, 0.2], seed=42)

# Es una buena práctica verificar el tamaño de cada conjunto
print(f"Total de clientes en el dataset: {dataset_final_para_modelar.count()}")
print(f"Clientes para entrenamiento: {training_set.count()}")
print(f"Clientes para validación: {validation_set.count()}")

# Ahora puedes guardar estos sets como tablas si lo deseas
training_set.write.mode("overwrite").saveAsTable("workspace.data.pedidos_distribucion_target_training_set")
validation_set.write.mode("overwrite").saveAsTable("workspace.data.pedidos_distribucion_target_validation_set")
dataset_final_para_modelar.write.mode("overwrite").saveAsTable("workspace.data.pedidos_distribucion_target")


print("¡Listo! La tabla 'prediccion_pedidos_distribucion_target' ha sido guardada en 'workspace.data'.")

In [0]:
from pyspark.sql.functions import col, count

# Cuenta el número total de clientes en el dataset de modelado
total_clientes_modelo = target.count()

# Agrupa por la variable target para ver la distribución
distribucion_target = target.groupBy("target_prox_trx_digital") \
    .agg(count("*").alias("numero_de_clientes")) \
    .withColumn(
        "porcentaje",
        (col("numero_de_clientes") / total_clientes_modelo) * 100
    )

display(distribucion_target)